In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

from dwave_simulator import DwaveSimulator

In [ ]:
sim = DwaveSimulator()

In [ ]:
small_problem = sim.generate_random_ising_problem(
    n=5,
    weight_min=-1.0,
    weight_max=1.0,
    random_seed=42
)

print("Small Ising problem:")
print(small_problem)

In [ ]:
small_eigenvalues_history, small_eigenvectors_history = sim.simulate_noisy_evolution(
    small_problem,
    nb_eigenvalues=5,
    noise_std=0.1,
    random_seed=42
)

print("Number of annealing steps:", len(small_eigenvalues_history))
print("Five lowest eigenvalues at the beginning:")
print(small_eigenvalues_history[0])
print("Five lowest eigenvalues at the end:")
print(small_eigenvalues_history[-1])

In [ ]:
sim.plot_eigenvalues(
    small_eigenvalues_history,
    title="Five lowest eigenvalues for a noisy 5-variable Ising instance"
)

In [ ]:
small_gap = sim.plot_spectral_gap(
    small_eigenvalues_history,
    title="Spectral gap for a noisy 5-variable Ising instance"
)

print("Minimum gap:", small_gap.min())
print("Step of minimum gap:", small_gap.argmin())

In [ ]:
sizes = [3, 4, 5, 6, 7, 8]
min_gaps = []

for n in sizes:
    problem_n = sim.generate_random_ising_problem(
        n=n,
        weight_min=-1.0,
        weight_max=1.0,
        random_seed=42
    )

    eig_hist_n, _ = sim.simulate_noisy_evolution(
        problem_n,
        nb_eigenvalues=5,
        noise_std=0.1,
        random_seed=42
    )

    eig_hist_n = np.array(eig_hist_n, dtype=float)
    gap_n = eig_hist_n[:, 1] - eig_hist_n[:, 0]
    min_gaps.append(gap_n.min())

plt.figure()
plt.plot(sizes, min_gaps, marker="o")
plt.xlabel("Number of variables")
plt.ylabel("Minimum spectral gap")
plt.title("Minimum spectral gap vs instance size (noisy)")
plt.grid(True)
plt.show()

for n, g in zip(sizes, min_gaps):
    print(f"n = {n} -> min gap = {g:.6f}")

In [ ]:
base_problem = sim.generate_random_ising_problem(
    n=5,
    weight_min=-1.0,
    weight_max=1.0,
    random_seed=123
)

scales = [0.25, 0.5, 1.0, 2.0]
scaled_min_gaps = []

for scale in scales:
    scaled_problem = sim.rescale_ising_problem(base_problem, scale)

    eig_hist_scaled, _ = sim.simulate_noisy_evolution(
        scaled_problem,
        nb_eigenvalues=5,
        noise_std=0.1,
        random_seed=42
    )

    eig_hist_scaled = np.array(eig_hist_scaled, dtype=float)
    gap_scaled = eig_hist_scaled[:, 1] - eig_hist_scaled[:, 0]
    scaled_min_gaps.append(gap_scaled.min())

plt.figure()
plt.plot(scales, scaled_min_gaps, marker="o")
plt.xlabel("Rescaling factor")
plt.ylabel("Minimum spectral gap")
plt.title("Impact of rescaling on the spectral gap (noisy)")
plt.grid(True)
plt.show()

for scale, g in zip(scales, scaled_min_gaps):
    print(f"scale = {scale} -> min gap = {g:.6f}")

## Discussion

For the noisy 5-variable Ising instance, the five lowest eigenvalues still evolve during the annealing process, but the presence of noise perturbs the spectrum at each step. As a result, the evolution is less regular than in the noiseless case.

The spectral gap remains an important indicator of the difficulty of the annealing process. When the gap becomes small, the system is more sensitive to perturbations, and the added noise can make it harder to remain in the ground state.

When comparing different instance sizes, the minimum spectral gap varies from one instance to another, but small gaps still indicate more difficult problems. In the noisy case, these small gaps are even more critical because the noise can further destabilize the evolution.

For the rescaling study, reducing the coefficients of the final Hamiltonian makes the useful energy scale smaller compared to the noise level. This can make the problem harder to solve in practice, because the noise has a stronger relative effect on the spectrum and on the spectral gap.

Overall, the noisy simulations show that noise can degrade the resolution of the problem, especially when the spectral gap is already small or when the Hamiltonian coefficients are strongly rescaled.